In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


### Barrel_Temp_Std 검증

In [ ]:
X_temp = X_cn7.copy()

temp_cols = [
    col for col in X_temp.columns
    if col.startswith("Barrel_Temperature")
]

X_temp["Barrel_Temp_Std"] = X_temp[temp_cols].std(
    axis=1
)

In [5]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate


# 1. Barrel_Temp_Std 생성
X_temp = X_cn7.copy()

temp_cols = [
    col for col in X_temp.columns
    if col.startswith("Barrel_Temperature")
]

X_temp["Barrel_Temp_Std"] = X_temp[temp_cols].std(axis=1)


# 2. Repeated Stratified CV
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

scoring = {
    "PR-AUC": "average_precision",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1"
}


# 3. Baseline vs 파생변수
datasets = {
    "Baseline": X_cn7,
    "Barrel_Temp_Std": X_temp
}

results = []

for feature_name, X_data in datasets.items():

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_validate(
        model,
        X_data,
        y_cn7,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Feature": feature_name,

        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),

        "Recall Mean": scores["test_Recall"].mean(),
        "Recall Std": scores["test_Recall"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Precision Std": scores["test_Precision"].std(),

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std()
    })


# 4. 결과
feature_cv_df = pd.DataFrame(results)

display(feature_cv_df)

,Feature,PR-AUC Mean,PR-AUC Std,Recall Mean,Recall Std,Precision Mean,Precision Std,F1 Mean,F1 Std
0,Baseline,0.497936,0.220785,0.34,0.233238,0.740000,0.427083,0.455619,0.287246
1,Barrel_Temp_Std,0.487214,0.207519,0.34,0.233238,0.726667,0.423950,0.450286,0.281992


단일 Hold-out 평가에서 Logistic Regression의 성능 향상을 보였던 Barrel_Temp_Std를 Repeated Stratified 5-Fold CV로 재검증하였다. 그 결과 PR-AUC는 0.4979에서 0.4872, F1은 0.4556에서 0.4503으로 소폭 감소했으며 Recall은 0.3400으로 동일하였다. 따라서 기존 단일 분할에서 관찰된 성능 향상은 반복 교차검증에서 재현되지 않았으며, Barrel_Temp_Std의 추가적인 예측 기여는 제한적인 것으로 판단하였다.

### Process_Time_Sum CV 검증

In [7]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate


# 1. Process_Time_Sum 생성
X_temp = X_cn7.copy()

X_temp["Process_Time_Sum"] = (
    X_temp["Injection_Time"]
    + X_temp["Filling_Time"]
    + X_temp["Plasticizing_Time"]
)


# 2. Repeated Stratified CV
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

scoring = {
    "PR-AUC": "average_precision",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1"
}


# 3. Baseline vs Process_Time_Sum
datasets = {
    "Baseline": X_cn7,
    "Process_Time_Sum": X_temp
}

results = []

for feature_name, X_data in datasets.items():

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_validate(
        model,
        X_data,
        y_cn7,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Feature": feature_name,

        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),

        "Recall Mean": scores["test_Recall"].mean(),
        "Recall Std": scores["test_Recall"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Precision Std": scores["test_Precision"].std(),

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std()
    })


feature_cv_df = pd.DataFrame(results)

display(feature_cv_df)

,Feature,PR-AUC Mean,PR-AUC Std,Recall Mean,Recall Std,Precision Mean,Precision Std,F1 Mean,F1 Std
0,Baseline,0.497936,0.220785,0.34,0.233238,0.74,0.427083,0.455619,0.287246
1,Process_Time_Sum,0.496346,0.220472,0.34,0.233238,0.74,0.427083,0.455619,0.287246


Process_Time_Sum을 Repeated Stratified 5-Fold CV로 재검증한 결과, Recall·Precision·F1은 Baseline과 동일하게 나타났으며 PR-AUC 역시 0.4979에서 0.4963으로 거의 변화하지 않았다. 따라서 전체 공정시간 정보를 합산한 파생변수는 Logistic Regression에 추가적인 판별 정보를 제공하지 못한 것으로 판단하였다.

### Screw_RPM_Mean CV 검증

In [8]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate


# 1. Screw_RPM_Mean 생성
X_temp = X_cn7.copy()

X_temp["Screw_RPM_Mean"] = (
    X_temp["Max_Screw_RPM"]
    + X_temp["Average_Screw_RPM"]
) / 2


# 2. CV 설정
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

scoring = {
    "PR-AUC": "average_precision",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1"
}


# 3. 모델
models_feature = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "LightGBM": LGBMClassifier(
        random_state=42,
        verbose=-1
    )
}


# 4. Baseline vs Screw_RPM_Mean
datasets = {
    "Baseline": X_cn7,
    "Screw_RPM_Mean": X_temp
}

results = []

for model_name, model in models_feature.items():

    for feature_name, X_data in datasets.items():

        scores = cross_validate(
            model,
            X_data,
            y_cn7,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )

        results.append({
            "Model": model_name,
            "Feature": feature_name,
            "PR-AUC Mean": scores["test_PR-AUC"].mean(),
            "PR-AUC Std": scores["test_PR-AUC"].std(),
            "Recall Mean": scores["test_Recall"].mean(),
            "Recall Std": scores["test_Recall"].std(),
            "Precision Mean": scores["test_Precision"].mean(),
            "Precision Std": scores["test_Precision"].std(),
            "F1 Mean": scores["test_F1"].mean(),
            "F1 Std": scores["test_F1"].std()
        })


feature_cv_df = pd.DataFrame(results)

display(feature_cv_df)

,Model,Feature,PR-AUC Mean,PR-AUC Std,Recall Mean,Recall Std,Precision Mean,Precision Std,F1 Mean,F1 Std
0,Logistic Regression,Baseline,0.497936,0.220785,0.34,0.233238,0.740000,0.427083,0.455619,0.287246
1,Logistic Regression,Screw_RPM_Mean,0.497599,0.220871,0.34,0.233238,0.740000,0.427083,0.455619,0.287246
2,LightGBM,Baseline,0.462618,0.188005,0.34,0.233238,0.386857,0.287991,0.336000,0.214165
3,LightGBM,Screw_RPM_Mean,0.467378,0.187976,0.34,0.233238,0.386857,0.287991,0.336000,0.214165


Screw_RPM_Mean을 Repeated Stratified 5-Fold CV로 검증한 결과, Logistic Regression에서는 Baseline과 거의 동일한 성능을 보였다. LightGBM에서는 PR-AUC가 0.4626에서 0.4674로 소폭 상승했으나 Recall, Precision, F1은 변화하지 않았다. 따라서 단일 Hold-out에서 관찰된 성능 향상은 반복 교차검증에서 뚜렷하게 재현되지 않았으며, Screw_RPM_Mean의 추가적인 예측 기여는 제한적인 것으로 판단하였다.

### Group CV 검증

#### 동일 X를 group id로 만들기

In [9]:
import pandas as pd

# 동일한 X 조합에 동일 Group ID 부여
group_id = pd.util.hash_pandas_object(
    X_cn7,
    index=False
)

print("전체 데이터:", len(X_cn7))
print("고유 공정조건:", group_id.nunique())
print("중복 공정조건 데이터:", len(X_cn7) - group_id.nunique())

전체 데이터: 1211
고유 공정조건: 606
중복 공정조건 데이터: 605


In [10]:
group_size = group_id.value_counts()

print("반복된 공정조건 그룹 수:", (group_size > 1).sum())
print("가장 많이 반복된 횟수:", group_size.max())

반복된 공정조건 그룹 수: 605
가장 많이 반복된 횟수: 2


#### group cv 가능한지 확인

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X_cn7, y_cn7, groups=group_id),
    start=1
):
    print(
        f"Fold {fold} | "
        f"Train: {len(train_idx)} "
        f"(불량 {y_cn7.iloc[train_idx].sum()}) | "
        f"Validation: {len(val_idx)} "
        f"(불량 {y_cn7.iloc[val_idx].sum()})"
    )

Fold 1 | Train: 969 (불량 13) | Validation: 242 (불량 4)
Fold 2 | Train: 971 (불량 14) | Validation: 240 (불량 3)
Fold 3 | Train: 967 (불량 14) | Validation: 244 (불량 3)
Fold 4 | Train: 969 (불량 13) | Validation: 242 (불량 4)
Fold 5 | Train: 968 (불량 14) | Validation: 243 (불량 3)


### 일반 CV vs Group CV

In [12]:
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_validate
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# 모델
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        verbose=-1
    ),

    "XGBoost": XGBClassifier(
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}


# 일반 Stratified CV
stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# 동일 X를 분리하는 Group CV
group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


scoring = {
    "PR-AUC": "average_precision",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1"
}


results = []

for name, model in models.items():

    # 일반 CV
    scores = cross_validate(
        model,
        X_cn7,
        y_cn7,
        cv=stratified_cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "CV": "Stratified",
        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),
        "Recall Mean": scores["test_Recall"].mean(),
        "Precision Mean": scores["test_Precision"].mean(),
        "F1 Mean": scores["test_F1"].mean()
    })


    # Group CV
    scores = cross_validate(
        model,
        X_cn7,
        y_cn7,
        cv=group_cv,
        groups=group_id,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "CV": "Group",
        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),
        "Recall Mean": scores["test_Recall"].mean(),
        "Precision Mean": scores["test_Precision"].mean(),
        "F1 Mean": scores["test_F1"].mean()
    })


group_compare_df = pd.DataFrame(results)

display(group_compare_df)

,Model,CV,PR-AUC Mean,PR-AUC Std,Recall Mean,Precision Mean,F1 Mean
0,Logistic Regression,Stratified,0.508069,0.203637,0.333333,0.800000,0.466667
1,Logistic Regression,Group,0.438554,0.272092,0.333333,0.600000,0.426667
2,Random Forest,Stratified,0.384048,0.183647,0.333333,0.416667,0.337143
3,Random Forest,Group,0.424200,0.305378,0.333333,0.600000,0.426667
4,LightGBM,Stratified,0.439214,0.207103,0.333333,0.416667,0.337143
5,LightGBM,Group,0.486700,0.297713,0.233333,0.400000,0.293333
6,XGBoost,Stratified,0.405317,0.134129,0.333333,0.416667,0.337143
7,XGBoost,Group,0.525124,0.290062,0.000000,0.000000,0.000000
8,CatBoost,Stratified,0.442742,0.151578,0.333333,0.416667,0.337143
9,CatBoost,Group,0.432116,0.278077,0.333333,0.600000,0.426667


동일 공정조건의 Train-Validation 중복이 모델 성능에 미치는 영향을 확인하기 위해 동일 설명변수 조합을 하나의 그룹으로 정의하고 Stratified Group 5-Fold CV를 수행하였다. Group CV 적용 후 Logistic Regression과 CatBoost의 PR-AUC는 감소한 반면 Random Forest, LightGBM, XGBoost는 증가하여 모든 모델에서 일관된 성능 하락은 관찰되지 않았다. 따라서 기존 Stratified CV의 성능이 동일 공정조건 중복에 의해 전반적으로 과대평가되었다고 판단하기는 어렵다. 다만 Group CV에서 모든 모델의 PR-AUC 표준편차가 증가하였으며, 이는 불량 표본이 17건으로 매우 적어 그룹 구성에 따라 평가 결과가 크게 변동할 수 있음을 보여준다.

파생변수 → 불균형 처리 → Threshold → 확률분포 → Group CV까지 확인했는데, 특정 기법 하나로 안정적인 성능 향상이 나타나지 않았음. 오히려 CN7의 핵심 한계는 불량 17건이라는 극단적인 희소성 + 일부 불량의 낮은 예측확률 + fold에 따른 높은 변동성으로 보는 게 맞음